In [47]:
import os

import ee
import geemap

from find_set_root import find_set_project_root
PROJECT_ROOT = find_set_project_root()
print(f"Project root found at: {PROJECT_ROOT}")
import utils.general_functions as ugf


DIR_RAW = os.path.join(PROJECT_ROOT, 'data', 'raw')
DIR_DERIVED = os.path.join(PROJECT_ROOT, 'data', 'derived')
ugf.dir_ensure([DIR_RAW, DIR_DERIVED])

GEE_PROJECT = 'ee-tymc5571-multi-disturbance'
GEE_PROJECT_DIR = 'projects/ee-tymc5571-multi-disturbance/assets'
DRIVE_FOLDER = "GEE_Exports_resilience_20260212"


#Prepare to use Earth Engine
ee.Authenticate(
    auth_mode='notebook',
    scopes=[
        'https://www.googleapis.com/auth/earthengine',
        'https://www.googleapis.com/auth/devstorage.read_write',
        'https://www.googleapis.com/auth/drive'
    ]
    #, force = True
)
ee.Initialize(project=GEE_PROJECT)

SERVICE_FILE = "C:/Users/tymc5571/dev/cbi-module/config/secrets/tymc5571-utils-project-692f27c034dd.json"  # Path to service account file

EXPORT_CRS = 'EPSG:5070'


Project root found at: C:\Users\tymc5571\dev\compound-disturbance-resilience
✅ Directory already exists: C:\Users\tymc5571\dev\compound-disturbance-resilience\data\raw
✅ Directory already exists: C:\Users\tymc5571\dev\compound-disturbance-resilience\data\derived


In [48]:
biotic = ee.Image("projects/ee-tymc5571-multi-disturbance/assets/biotic_gridded_1km_all_years_severity")
relaxed_forest = ee.Image("projects/ee-tymc5571-cbi-module/assets/relaxed_forest_1985_2024")

states = ee.FeatureCollection("TIGER/2018/States")
western_states_list = ['WA','OR','CA','ID','NV','MT','WY','UT','CO','AZ','NM']
west_geom = states.filter(
    ee.Filter.inList('STUSPS', western_states_list)
).geometry()

biotic_proj = biotic.projection()  # EPSG:5070 1km
biotic_scale = biotic_proj.nominalScale()  # should be 1000

BIOTIC_CRS = "EPSG:5070"
BIOTIC_TRANSFORM = [1000, 0, -2375000, 0, -1000, 3189999]

west_region_5070 = ee.Geometry(west_geom).transform(biotic_proj, 1).bounds(1)



In [49]:

# West region in lon/lat (export-safe, map-safe)
west_region_ll = ee.Geometry(west_geom).bounds(1)

def make_tiles_ll(rect_ll: ee.Geometry, nx=3, ny=3):
    rect_ll = ee.Geometry(rect_ll).bounds(1)
    coords = ee.List(rect_ll.coordinates().get(0))
    xs = coords.map(lambda p: ee.Number(ee.List(p).get(0)))
    ys = coords.map(lambda p: ee.Number(ee.List(p).get(1)))
    xmin = ee.Number(xs.reduce(ee.Reducer.min()))
    xmax = ee.Number(xs.reduce(ee.Reducer.max()))
    ymin = ee.Number(ys.reduce(ee.Reducer.min()))
    ymax = ee.Number(ys.reduce(ee.Reducer.max()))

    dx = xmax.subtract(xmin).divide(nx)
    dy = ymax.subtract(ymin).divide(ny)

    tiles = []
    for ix in range(nx):
        for iy in range(ny):
            x0 = xmin.add(dx.multiply(ix))
            x1 = xmin.add(dx.multiply(ix + 1))
            y0 = ymin.add(dy.multiply(iy))
            y1 = ymin.add(dy.multiply(iy + 1))
            tiles.append(ee.Geometry.Rectangle([x0, y0, x1, y1], proj="EPSG:4326", geodesic=False))
    return tiles

tiles = make_tiles_ll(west_region_ll, nx=3, ny=3)


START_YEAR = 1997
N_BANDS = 27
EPS = 1e-6

def year_to_biotic_band(year: int) -> str:
    return f"b{year - START_YEAR + 1}"

def forest_fraction_tile(year_prev: int, tile: ee.Geometry) -> ee.Image:
    # Work *inside tile* before any projection alignment
    f01 = (relaxed_forest
           .select(f"forest_{year_prev}")
           .unmask(0)
           .toFloat()
           .clip(tile))

    # Aggregate to 1km fraction in the forest's own projection,
    # then reproject to the biotic grid at 1km (inside the tile)
    ffrac = (f01
        .reduceResolution(
            reducer=ee.Reducer.mean(),
            maxPixels=8192,
            bestEffort=True
        )
        .reproject(crs=BIOTIC_CRS, crsTransform=BIOTIC_TRANSFORM)
        .rename(f"forestfrac_{year_prev}")
    )
    return ffrac

def normalized_biotic_tile(year: int, tile: ee.Geometry) -> ee.Image:
    b = (biotic
         .select(year_to_biotic_band(year))
         .toFloat()
         .clip(tile))

    ffrac = forest_fraction_tile(year - 1, tile)

    # Mask out zero-forest pixels to avoid eps blowups, then cap at 100
    norm = (b
        .divide(ffrac.max(EPS))
        .updateMask(ffrac.gt(0))
        .clamp(0, 100)
        .rename(f"biotic_norm_{year}")
    )
    return norm

def normalized_stack_for_tile(tile: ee.Geometry) -> ee.Image:
    years = list(range(START_YEAR, START_YEAR + N_BANDS))  # 1997..2023
    bands = [normalized_biotic_tile(y, tile) for y in years]
    return ee.Image.cat(bands).reproject(crs=BIOTIC_CRS, crsTransform=BIOTIC_TRANSFORM)



In [50]:
import ee
import geemap
ee.Initialize()

TEST_YEAR = 2015
FOREST_YEAR = TEST_YEAR - 1

# Front Range-ish bbox (Cheyenne to Pueblo; foothills to eastern plains)
front_range_ll = ee.Geometry.Rectangle(
    [-106.25, 38.6, -104.0, 41.3],
    proj="EPSG:4326",
    geodesic=False
)

m = geemap.Map(center=[40.0, -105.2], zoom=7)

# 1) Raw forest (previous year): binary mask (1s, elsewhere masked)
# Use selfMask() so only 1s draw (cleaner); unmask(0) would draw background as 0.
raw_forest = relaxed_forest.select(f"forest_{FOREST_YEAR}").clip(front_range_ll).selfMask()

m.addLayer(
    raw_forest,
    {"min": 0, "max": 1, "palette": ["00FF00"]},  # green for presence
    f"Forest raw (1s) {FOREST_YEAR}"
)

# 2) Forest fraction at 1 km on biotic grid (previous year)
forest_frac = forest_fraction_tile(FOREST_YEAR, front_range_ll)

m.addLayer(
    forest_frac,
    {"min": 0, "max": 1, "palette": ["black", "white"]},
    f"Forest fraction (1km) {FOREST_YEAR}"
)

# 3) Raw biotic (one band)
biotic_raw = biotic.select(year_to_biotic_band(TEST_YEAR)).clip(front_range_ll)

m.addLayer(
    biotic_raw,
    {"min": 0, "max": 20, "palette": ["black", "white"]},  # adjust min/max to your biotic range
    f"Biotic raw {TEST_YEAR}"
)

# 4) Normalized biotic (capped at 100)
biotic_norm_preview = normalized_biotic_tile(TEST_YEAR, front_range_ll)

m.addLayer(
    biotic_norm_preview,
    {"min": 0, "max": 20, "palette": ["black", "white"]},
    f"Biotic normalized {TEST_YEAR} (cap 100)"
)

m.addLayer(front_range_ll, {}, "Front Range AOI", False)
m


Map(center=[40.0, -105.2], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

In [51]:

# --- Make a FeatureCollection of tile bounding boxes ---
tile_fc = ee.FeatureCollection([
    ee.Feature(t, {"tile_id": i}) for i, t in enumerate(tiles)
])

# Style: outlines only
tile_outline = tile_fc.style(
    color="FF0000",      # red outline
    fillColor="00000000",# transparent fill
    width=2
)

west_outline = ee.FeatureCollection([ee.Feature(west_region_5070, {"name": "west_region_5070"})]).style(
    color="0000FF",      # blue outline
    fillColor="00000000",
    width=3
)

# --- Visualize ---
m_tiles = geemap.Map(center=[40.0, -105.2], zoom=4)

m_tiles.addLayer(west_outline, {}, "Export region (west_region_5070) outline")
m_tiles.addLayer(tile_outline, {}, "Export tiles (3x3) outlines")

# Optional: add the tile geometries themselves (clickable)
m_tiles.addLayer(tile_fc, {}, "Tiles (clickable)", False)

m_tiles

Map(center=[40.0, -105.2], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

In [52]:
for i, tile in enumerate(tiles):
    img = normalized_stack_for_tile(tile)  # tile is now lon/lat; that's OK

    task = ee.batch.Export.image.toDrive(
        image=img,
        description=f"biotic_norm_relaxedprevforest_tile_{i:02d}",
        folder=DRIVE_FOLDER,
        fileNamePrefix=f"biotic_norm_relaxedprevforest_tile_{i:02d}",
        region=tile,                 # lon/lat tile
        crs=BIOTIC_CRS,              # EPSG:5070
        crsTransform=BIOTIC_TRANSFORM,
        maxPixels=1e13
    )
    task.start()
    print("Started tile", i)


Started tile 0
Started tile 1
Started tile 2
Started tile 3
Started tile 4
Started tile 5
Started tile 6
Started tile 7
Started tile 8


# EXPORT & HELPERS

In [ ]:
# # EXPORT to asset first, allowing lazy CRS management

# assetID = GEE_PROJECT_DIR + "/biotic_forestnorm"

# # Export to asset
# task_asset = ee.batch.Export.image.toAsset(
#     image=biotic_norm_masked_wsum,
#     description="biotic_forestnorm" + "_toAsset",
#     assetId=assetID,
#     region=western_states.geometry(),
#     scale=1000,
#     maxPixels=1e13
# )

# task_asset.start()



In [11]:
# Re-load and export from pyramid-backed asset, forcing CRS to 5070 but hopefully still allowing sharding

# biotic_asset = ee.Image(assetID).select

# biotic_no_sum = biotic_asset.select(
#     biotic_asset.bandNames().remove("biotic_forestnorm_timeseries_sum")
# )

# Export to Drive; needs tiling due to the forced projection piece
task_drive = ee.batch.Export.image.toDrive(
    image=biotic,
    description="biotic_test" + "_toDrive",
    folder=DRIVE_FOLDER,
    region=western_states.geometry(),
    scale=1000,
    crs=EXPORT_CRS,
    maxPixels=1e13
)

task_drive.start()

In [5]:
import ee

EXPORT_CRS = "EPSG:5070"
SCALE = 1000  # meters

# Robust dissolve of the region
west_geom = western_states.union().geometry()

# Export the image you actually want (includes sum band)
img_to_export = biotic_forestnorm_masked.clip(west_geom)

def bbox_5070(geom):
    """Return bbox (xmin, ymin, xmax, ymax) in EPSG:5070 as client-side floats."""
    b = geom.transform(EXPORT_CRS, 1).bounds(proj=EXPORT_CRS)
    coords = ee.List(ee.List(b.coordinates().get(0)))
    # Rectangle ring: (xmin,ymin)->(xmax,ymin)->(xmax,ymax)->(xmin,ymax)->(xmin,ymin)
    xmin = ee.Number(ee.List(coords.get(0)).get(0))
    ymin = ee.Number(ee.List(coords.get(0)).get(1))
    xmax = ee.Number(ee.List(coords.get(2)).get(0))
    ymax = ee.Number(ee.List(coords.get(2)).get(1))
    return [v.getInfo() for v in [xmin, ymin, xmax, ymax]]

xmin, ymin, xmax, ymax = bbox_5070(west_geom)

xmid = (xmin + xmax) / 2.0
ymid = (ymin + ymax) / 2.0

tiles = [
    ("x00_y00", [xmin, ymin, xmid, ymid]),  # SW
    ("x01_y00", [xmid, ymin, xmax, ymid]),  # SE
    ("x00_y01", [xmin, ymid, xmid, ymax]),  # NW
    ("x01_y01", [xmid, ymid, xmax, ymax]),  # NE
]

tasks = []
for suffix, (x0, y0, x1, y1) in tiles:
    tile_geom = ee.Geometry.Rectangle([x0, y0, x1, y1], proj=EXPORT_CRS, geodesic=False)

    desc = f"biotic_forestnorm_5070_1km_{suffix}"

    task = ee.batch.Export.image.toDrive(
        image=img_to_export,
        description=desc,
        folder=DRIVE_FOLDER,
        fileNamePrefix=desc,
        region=tile_geom,
        crs=EXPORT_CRS,
        scale=SCALE,
        maxPixels=1e13
    )
    task.start()
    tasks.append(task)

print(f"Started {len(tasks)} exports (forced 2x2 tiling).")


Started 4 exports (forced 2x2 tiling).


In [6]:
import ee

EXPORT_CRS = "EPSG:5070"
SCALE = 1000  # meters

west_geom = western_states.union().geometry()

img_to_export = biotic_forestnorm_masked.clip(west_geom)

def bbox_5070(geom):
    b = geom.transform(EXPORT_CRS, 1).bounds(proj=EXPORT_CRS)
    coords = ee.List(ee.List(b.coordinates().get(0)))
    xmin = ee.Number(ee.List(coords.get(0)).get(0))
    ymin = ee.Number(ee.List(coords.get(0)).get(1))
    xmax = ee.Number(ee.List(coords.get(2)).get(0))
    ymax = ee.Number(ee.List(coords.get(2)).get(1))
    return [v.getInfo() for v in [xmin, ymin, xmax, ymax]]

xmin, ymin, xmax, ymax = bbox_5070(west_geom)

xmid = (xmin + xmax) / 2.0
ymid = (ymin + ymax) / 2.0

tiles = [
    ("SW", [xmin, ymin, xmid, ymid]),
    ("SE", [xmid, ymin, xmax, ymid]),
    ("NW", [xmin, ymid, xmid, ymax]),
    ("NE", [xmid, ymid, xmax, ymax]),
]

print("\nFull bbox (5070 meters):")
print(f"xmin={xmin}, ymin={ymin}, xmax={xmax}, ymax={ymax}")

print("\nTile extents:")
for name, (x0, y0, x1, y1) in tiles:
    w = x1 - x0
    h = y1 - y0
    print(f"{name}:")
    print(f"  xmin={x0:.0f}, ymin={y0:.0f}, xmax={x1:.0f}, ymax={y1:.0f}")
    print(f"  width_km={w/1000:.1f}, height_km={h/1000:.1f}")



Full bbox (5070 meters):
xmin=-2361581.263023, ymin=991812.651244, xmax=-504621.14299, ymax=3177422.386741

Tile extents:
SW:
  xmin=-2361581, ymin=991813, xmax=-1433101, ymax=2084618
  width_km=928.5, height_km=1092.8
SE:
  xmin=-1433101, ymin=991813, xmax=-504621, ymax=2084618
  width_km=928.5, height_km=1092.8
NW:
  xmin=-2361581, ymin=2084618, xmax=-1433101, ymax=3177422
  width_km=928.5, height_km=1092.8
NE:
  xmin=-1433101, ymin=2084618, xmax=-504621, ymax=3177422
  width_km=928.5, height_km=1092.8


In [7]:
m = geemap.Map()

for name, (x0,y0,x1,y1) in tiles:
    geom = ee.Geometry.Rectangle([x0,y0,x1,y1], proj=EXPORT_CRS, geodesic=False)
    m.addLayer(geom, {}, name)

m

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [38]:
import os
import shutil
import tempfile
import contextlib
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Optional, List
import time, random

from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from googleapiclient.http import MediaIoBaseDownload

# GDAL bindings
from osgeo import gdal


def list_all_files_in_folder(
    service,
    folder_id: str,
    description: Optional[str] = None,
    *,
    include_all_drives: bool = True,
    drive_id: Optional[str] = None,
    page_size: int = 1000,
) -> List[dict]:
    """
    List ALL files in a Drive folder, with pagination.
    Filters by description substring and .tif/.TIF extension.
    Returns a list of dicts with at least {'id','name'}.
    """
    # Query: in folder, not trashed, optional name contains 'description',
    # and extension contains '.tif' or '.TIF'
    name_filter = []
    if description:
        name_filter.append(f"(name contains '{description}')")
    ext_filter = "((name contains '.tif') or (name contains '.TIF'))"
    q_parts = [f"'{folder_id}' in parents", "trashed=false", ext_filter]
    if name_filter:
        q_parts.append(" and ".join(name_filter))
    q = " and ".join(q_parts)

    kwargs = dict(
        q=q,
        pageSize=page_size,
        fields="nextPageToken, files(id, name)",
        orderBy="name",
    )
    if include_all_drives:
        kwargs.update(dict(supportsAllDrives=True, includeItemsFromAllDrives=True))
        if drive_id:
            kwargs.update(dict(corpora="drive", driveId=drive_id))
        else:
            kwargs.update(dict(corpora="allDrives"))
    else:
        kwargs.update(dict(corpora="user"))

    files, page_token = [], None
    while True:
        resp = service.files().list(pageToken=page_token, **kwargs).execute()
        files.extend(resp.get("files", []))
        page_token = resp.get("nextPageToken")
        if not page_token:
            break
    return files

def _download_file_from_drive(file_id, file_name, temp_dir, service_account_file):
    try:
        # Authenticate service account
        SCOPES = ['https://www.googleapis.com/auth/drive.readonly']
        credentials = service_account.Credentials.from_service_account_file(
            os.path.abspath(service_account_file), scopes=SCOPES
        )
        service = build('drive', 'v3', credentials=credentials)

        request = service.files().get_media(fileId=file_id)
        local_path = os.path.join(temp_dir, file_name)
        with open(local_path, 'wb') as f:
            downloader = MediaIoBaseDownload(f, request)
            done = False
            while not done:
                status, done = downloader.next_chunk()
        time.sleep(random.uniform(0.5, 1.5))  # Random delay
        print(f"Downloaded {file_name}")
        return local_path
    except Exception as e:
        print(f"Error downloading {file_name}: {e}")
        return None


def download_merge_from_drive_tif(
    description: str,
    local_filename: str,
    drive_folder: str,
    service_account_file: str,
    *,
    # Output & mosaic controls
    compress: str = "DEFLATE",          # DEFLATE|LZW|ZSTD|JPEG
    predictor: Optional[int] = None,     # 2 for integers, 3 for floats; None = let GDAL decide
    blocksize: int = 512,                # tile size for output GeoTIFF
    bigtiff: str = "YES",                # YES|IF_SAFER
    dst_srs: Optional[str] = None,       # e.g., "EPSG:5070" (if reprojection needed)
    xres: Optional[float] = None,        # set both xres & yres to enforce pixel size (e.g., 30)
    yres: Optional[float] = None,
    dst_nodata: Optional[float] = None,  # set if you need a specific nodata in output
    # Drive / listing controls
    drive_id: Optional[str] = None,      # if folder is in a Shared Drive, you can pass the Drive ID
    include_all_drives: bool = True,     # search across Shared Drives
    # Behavior controls
    check_existing: bool = True,
    n_workers: int = 4,
    verbose: bool = True,
) -> str:
    """
    Download all Drive .tif/.TIF files whose names contain `description` from `drive_folder`,
    then build a VRT and stream a mosaicked GeoTIFF to `local_filename` (no in-memory merge).
    """

    # --- Fast exit if we already have the target ---
    local_filename = os.path.abspath(local_filename)
    if check_existing and os.path.exists(local_filename):
        if verbose:
            print(f"File already exists: {local_filename}")
        return local_filename

    # --- Temp workspace ---
    temp_dir = tempfile.mkdtemp()
    vrt_path = os.path.join(temp_dir, "mosaic.vrt")
    downloaded_paths: List[str] = []

    # --- Build Drive client ---
    SCOPES = ["https://www.googleapis.com/auth/drive.readonly"]
    cred_path = os.path.abspath(service_account_file)
    credentials = service_account.Credentials.from_service_account_file(cred_path, scopes=SCOPES)
    service = build("drive", "v3", credentials=credentials)

    # Convenience flags for Shared Drives
    list_kwargs = dict(
        fields="nextPageToken, files(id, name, parents)",
        supportsAllDrives=True,
        includeItemsFromAllDrives=True
    ) if include_all_drives else dict(fields="nextPageToken, files(id, name, parents)")

    try:
        # --- Locate the folder (paginate just in case) ---
        folder_q = (
            f"name='{drive_folder}' and mimeType='application/vnd.google-apps.folder' and trashed=false"
        )
        folder_kwargs = dict(
            q=folder_q,
            pageSize=1000,
            corpora=("drive" if drive_id else "allDrives") if include_all_drives else "user",
            driveId=drive_id,
            **list_kwargs
        )

        folders, page_token = [], None
        while True:
            resp = service.files().list(pageToken=page_token, **folder_kwargs).execute()
            folders.extend(resp.get("files", []))
            page_token = resp.get("nextPageToken")
            if not page_token:
                break

        if not folders:
            raise FileNotFoundError(
                f"Folder '{drive_folder}' not found. "
                "Ensure it exists and is shared with the service account "
                f"({credentials.service_account_email}). "
                "If it is on a Shared Drive, also ensure the service account has at least Viewer."
            )
        folder_id = folders[0]["id"]

        # --- List ALL matching .tif files in that folder (paginated) ---
        files = list_all_files_in_folder(
            service,
            folder_id,
            description=description,
            include_all_drives=include_all_drives,
            drive_id=drive_id,
            page_size=1000
        )

        if not files:
            raise FileNotFoundError(
                f"No matching .tif/.TIF files found for '{description}' in '{drive_folder}'. "
                "Check filenames and sharing."
            )

        if verbose:
            print(f"Found {len(files)} file(s). Starting download to {temp_dir} ...")

        # --- Download the files (parallel if requested) ---
        if n_workers <= 1:
            for f in files:
                path = _download_file_from_drive(f["id"], f["name"], temp_dir, service_account_file)
                if path:
                    downloaded_paths.append(path)
        else:
            with ThreadPoolExecutor(max_workers=n_workers) as ex:
                futures = [
                    ex.submit(_download_file_from_drive, f["id"], f["name"], temp_dir, service_account_file)
                    for f in files
                ]
                for fut in as_completed(futures):
                    p = fut.result()
                    if p:
                        downloaded_paths.append(p)

        if not downloaded_paths:
            raise RuntimeError("Downloads finished but no files were saved.")

        if verbose:
            print(f"Building VRT from {len(downloaded_paths)} tile(s) ...")

        # --- Build VRT (lightweight, no data copy) ---
        vrt_options = {}
        if dst_nodata is not None:
            vrt_options.update(dict(srcNodata=dst_nodata, VRTNodata=dst_nodata))

        vrt = gdal.BuildVRT(vrt_path, downloaded_paths, **vrt_options)
        if vrt is None:
            raise RuntimeError("GDAL BuildVRT failed.")
        vrt = None  # flush

        # --- Stream out to GeoTIFF (no giant array in RAM) ---
        co = [
            f"BIGTIFF={bigtiff}",
            "TILED=YES",
            f"COMPRESS={compress.upper()}",
            f"BLOCKXSIZE={int(blocksize)}",
            f"BLOCKYSIZE={int(blocksize)}",
            "OVERVIEWS=AUTO", #pyramids
            "RESAMPLING=AVERAGE",
            "NUM_THREADS=ALL_CPUS"
        ]
        if predictor is not None:
            co.append(f"PREDICTOR={int(predictor)}")

        needs_warp = any(v is not None for v in (dst_srs, xres, yres, dst_nodata))
        if needs_warp:
            if verbose:
                print("Warping (streamed) to target grid...")
            warp_kwargs = dict(
                format="GTiff",
                creationOptions=co,
                multithread=True,
            )
            if dst_srs:
                warp_kwargs["dstSRS"] = dst_srs
            if xres and yres:
                warp_kwargs["xRes"] = float(xres)
                warp_kwargs["yRes"] = float(yres)
                warp_kwargs["targetAlignedPixels"] = True
            if dst_nodata is not None:
                warp_kwargs["dstNodata"] = dst_nodata

            out_ds = gdal.Warp(local_filename, vrt_path, **warp_kwargs)
            if out_ds is None:
                raise RuntimeError("GDAL Warp failed.")
            out_ds = None
        else:
            if verbose:
                print("Translating VRT to tiled, compressed GeoTIFF (streamed)...")
            out_ds = gdal.Translate(local_filename, vrt_path, creationOptions=co)
            if out_ds is None:
                raise RuntimeError("GDAL Translate failed.")
            out_ds = None

        if verbose:
            print(f"Final mosaicked GeoTIFF saved to: {local_filename}")
        return local_filename

    except HttpError as e:
        msg = str(e)
        if "accessNotConfigured" in msg or "has not been used in project" in msg:
            raise RuntimeError(
                "Google Drive API is disabled for this service account’s project.\n"
                "Enable the **Google Drive API** in GCP → APIs & Services → Library, then retry."
            ) from e
        raise
    except KeyboardInterrupt:
        print("Interrupted by user. Cleaning up and exiting.")
        raise
    finally:
        # Best-effort cleanup
        try:
            shutil.rmtree(temp_dir)
            if verbose:
                print(f"Cleaned up temporary directory: {temp_dir}")
        except Exception as cleanup_err:
            print(f"Error during cleanup: {cleanup_err}")


def download_merge_from_drive_cog(
    description: str,
    local_filename: str,
    drive_folder: str,
    service_account_file: str,
    *,
    # Output & mosaic controls
    compress: str = "DEFLATE",          # DEFLATE|LZW|ZSTD|JPEG
    predictor: Optional[int] = None,     # 2 for integers, 3 for floats; None = let GDAL decide
    blocksize: int = 512,                # tile size for output COG (both x and y)
    bigtiff: str = "YES",                # YES|IF_SAFER
    dst_srs: Optional[str] = None,       # e.g., "EPSG:5070" (if reprojection needed)
    xres: Optional[float] = None,        # set both xres & yres to enforce pixel size (e.g., 30)
    yres: Optional[float] = None,
    dst_nodata: Optional[float] = None,  # set if you need a specific nodata in output
    # Drive / listing controls
    drive_id: Optional[str] = None,      # if folder is in a Shared Drive, you can pass the Drive ID
    include_all_drives: bool = True,     # search across Shared Drives
    # Behavior controls
    check_existing: bool = True,
    n_workers: int = 4,
    verbose: bool = True,
) -> str:
    """
    Download all Drive .tif/.TIF files whose names contain `description` from `drive_folder`,
    then build a VRT and stream a mosaicked GeoTIFF to `local_filename` (no in-memory merge).
    """

    # --- Fast exit if we already have the target ---
    local_filename = os.path.abspath(local_filename)
    if check_existing and os.path.exists(local_filename):
        if verbose:
            print(f"File already exists: {local_filename}")
        return local_filename

    # --- Temp workspace ---
    temp_dir = tempfile.mkdtemp()
    vrt_path = os.path.join(temp_dir, "mosaic.vrt")
    downloaded_paths: List[str] = []

    # --- Build Drive client ---
    SCOPES = ["https://www.googleapis.com/auth/drive.readonly"]
    cred_path = os.path.abspath(service_account_file)
    credentials = service_account.Credentials.from_service_account_file(cred_path, scopes=SCOPES)
    service = build("drive", "v3", credentials=credentials)

    # Convenience flags for Shared Drives
    list_kwargs = dict(
        fields="nextPageToken, files(id, name, parents)",
        supportsAllDrives=True,
        includeItemsFromAllDrives=True
    ) if include_all_drives else dict(fields="nextPageToken, files(id, name, parents)")

    try:
        # --- Locate the folder (paginate just in case) ---
        folder_q = (
            f"name='{drive_folder}' and mimeType='application/vnd.google-apps.folder' and trashed=false"
        )
        folder_kwargs = dict(
            q=folder_q,
            pageSize=1000,
            corpora=("drive" if drive_id else "allDrives") if include_all_drives else "user",
            driveId=drive_id,
            **list_kwargs
        )

        folders, page_token = [], None
        while True:
            resp = service.files().list(pageToken=page_token, **folder_kwargs).execute()
            folders.extend(resp.get("files", []))
            page_token = resp.get("nextPageToken")
            if not page_token:
                break

        if not folders:
            raise FileNotFoundError(
                f"Folder '{drive_folder}' not found. "
                "Ensure it exists and is shared with the service account "
                f"({credentials.service_account_email}). "
                "If it is on a Shared Drive, also ensure the service account has at least Viewer."
            )
        folder_id = folders[0]["id"]

        # --- List ALL matching .tif files in that folder (paginated) ---
        files = list_all_files_in_folder(
            service,
            folder_id,
            description=description,
            include_all_drives=include_all_drives,
            drive_id=drive_id,
            page_size=1000
        )

        if not files:
            raise FileNotFoundError(
                f"No matching .tif/.TIF files found for '{description}' in '{drive_folder}'. "
                "Check filenames and sharing."
            )

        if verbose:
            print(f"Found {len(files)} file(s). Starting download to {temp_dir} ...")

        # --- Download the files (parallel if requested) ---
        if n_workers <= 1:
            for f in files:
                path = _download_file_from_drive(f["id"], f["name"], temp_dir, service_account_file)
                if path:
                    downloaded_paths.append(path)
        else:
            with ThreadPoolExecutor(max_workers=n_workers) as ex:
                futures = [
                    ex.submit(_download_file_from_drive, f["id"], f["name"], temp_dir, service_account_file)
                    for f in files
                ]
                for fut in as_completed(futures):
                    p = fut.result()
                    if p:
                        downloaded_paths.append(p)

        if not downloaded_paths:
            raise RuntimeError("Downloads finished but no files were saved.")

        if verbose:
            print(f"Building VRT from {len(downloaded_paths)} tile(s) ...")

        # --- Build VRT (lightweight, no data copy) ---
        vrt_options = {}
        if dst_nodata is not None:
            vrt_options.update(dict(srcNodata=dst_nodata, VRTNodata=dst_nodata))

        vrt = gdal.BuildVRT(vrt_path, downloaded_paths, **vrt_options)
        if vrt is None:
            raise RuntimeError("GDAL BuildVRT failed.")
        vrt = None  # flush

        # --- Stream out to Cloud-Optimized GeoTIFF (COG) ---
        # COG creation options (driver: "COG")
        co_cog = [
            f"COMPRESS={compress.upper()}",     # e.g., DEFLATE|LZW|ZSTD|JPEG
            f"BLOCKSIZE={int(blocksize)}",      # one value for both X/Y, typical 512
            f"NUM_THREADS={'ALL_CPUS'}",
            f"BIGTIFF={bigtiff}",               # YES|IF_SAFER
            "OVERVIEW_RESAMPLING=AVERAGE",      # build internal overviews with this kernel
            # Optional: "SPARSE_OK=YES",       # can help on sparse rasters
        ]
        if predictor is not None:
            co_cog.append(f"PREDICTOR={int(predictor)}")  # 2=int, 3=float (for DEFLATE/LZW)

        needs_warp = any(v is not None for v in (dst_srs, xres, yres, dst_nodata))

        if needs_warp:
            if verbose:
                print("Warping to target grid (VRT) and writing COG...")
            # Warp to a VRT (no data copy), then translate that VRT → COG
            warped_vrt = os.path.join(temp_dir, "warped.vrt")
            warp_kwargs = dict(
                format="VRT",           # keep it as VRT to avoid creating a big intermediate
                multithread=True,
            )
            if dst_srs:
                warp_kwargs["dstSRS"] = dst_srs
            if xres and yres:
                warp_kwargs["xRes"] = float(xres)
                warp_kwargs["yRes"] = float(yres)
                warp_kwargs["targetAlignedPixels"] = True
            if dst_nodata is not None:
                warp_kwargs["dstNodata"] = dst_nodata

            out_warp = gdal.Warp(warped_vrt, vrt_path, **warp_kwargs)
            if out_warp is None:
                raise RuntimeError("GDAL Warp (to VRT) failed.")
            out_warp = None

            # Final: VRT → COG
            out_ds = gdal.Translate(
                local_filename, warped_vrt,
                format="COG",
                creationOptions=co_cog
            )
            if out_ds is None:
                raise RuntimeError("GDAL Translate to COG failed.")
            out_ds = None

        else:
            if verbose:
                print("Translating VRT directly to COG (internal overviews)...")
            out_ds = gdal.Translate(
                local_filename, vrt_path,
                format="COG",
                creationOptions=co_cog
            )
            if out_ds is None:
                raise RuntimeError("GDAL Translate to COG failed.")
            out_ds = None

        if verbose:
            print(f"Final COG saved to: {local_filename}")
        return local_filename

    except HttpError as e:
        msg = str(e)
        if "accessNotConfigured" in msg or "has not been used in project" in msg:
            raise RuntimeError(
                "Google Drive API is disabled for this service account’s project.\n"
                "Enable the **Google Drive API** in GCP → APIs & Services → Library, then retry."
            ) from e
        raise
    except KeyboardInterrupt:
        print("Interrupted by user. Cleaning up and exiting.")
        raise
    finally:
        # Best-effort cleanup
        try:
            shutil.rmtree(temp_dir)
            if verbose:
                print(f"Cleaned up temporary directory: {temp_dir}")
        except Exception as cleanup_err:
            print(f"Error during cleanup: {cleanup_err}")

In [40]:
download_merge_from_drive_tif(
    description="biotic_norm_relaxedprevforest",
    local_filename=os.path.join(DIR_DERIVED, "biotic_relaxedforestnorm.tif"),
    drive_folder=DRIVE_FOLDER,
    service_account_file=SERVICE_FILE,
    compress="DEFLATE",         # same as 'deflate', case-insensitive
    predictor=3,                # 3 = best for float32 data (like indices or continuous values)
    blocksize=512,              # tile size; 256–512 is good default
    bigtiff="YES",              # ensures large mosaics don’t overflow 4 GB
    check_existing=True,
    n_workers=8,                # parallel downloads from Drive
    include_all_drives=True,    # safer if GEE_Exports is on a Shared Drive
    verbose=True
)

Found 9 file(s). Starting download to C:\Users\tymc5571\AppData\Local\Temp\tmpqi5foi9n ...
Downloaded biotic_norm_relaxedprevforest_tile_03.tif
Downloaded biotic_norm_relaxedprevforest_tile_00.tif
Downloaded biotic_norm_relaxedprevforest_tile_02.tif
Downloaded biotic_norm_relaxedprevforest_tile_06.tif
Downloaded biotic_norm_relaxedprevforest_tile_07.tif
Downloaded biotic_norm_relaxedprevforest_tile_04.tif
Downloaded biotic_norm_relaxedprevforest_tile_01.tif
Downloaded biotic_norm_relaxedprevforest_tile_05.tif
Downloaded biotic_norm_relaxedprevforest_tile_08.tif
Building VRT from 9 tile(s) ...
Translating VRT to tiled, compressed GeoTIFF (streamed)...
Final mosaicked GeoTIFF saved to: C:\Users\tymc5571\dev\compound-disturbance-resilience\data\derived\biotic_relaxedforestnorm.tif
Cleaned up temporary directory: C:\Users\tymc5571\AppData\Local\Temp\tmpqi5foi9n


'C:\\Users\\tymc5571\\dev\\compound-disturbance-resilience\\data\\derived\\biotic_relaxedforestnorm.tif'